# Laboratorio 2

In [5]:
import openpyxl

## a. Implemente la estructura de nodo descrita en la presentación de la clase.

In [7]:
class TreeNode:
    def __init__(self, name, heuristic=None):
        self.name = name
        self.parent = None
        self.children = []
        self.heuristic = heuristic
        self.cost = []

    def add_child(self, child_node, cost):
        self.children.append(child_node)
        self.cost.append(cost)
        child_node.parent = self


class Tree:
    def __init__(self):
        self.nodes = {}   # name -> TreeNode
        self.order = []   # mantiene orden de inserción
        self.root = None

    def add_node(self, name, heuristic=None):
        if name not in self.nodes:
            node = TreeNode(name, heuristic)
            self.nodes[name] = node
            self.order.append(node)
        else:
            self.nodes[name].heuristic = heuristic
        return self.nodes[name]

    def add_edge(self, parent_name, child_name, cost):
        parent = self.nodes.get(parent_name)
        child = self.nodes.get(child_name)
        if parent and child:
            parent.add_child(child, cost)

    def set_root(self):
        roots = [n for n in self.order if n.parent is None]
        self.root = roots[0] if roots else (self.order[0] if self.order else None)

    # Compatibilidad con celdas ya escritas
    def __iter__(self):
        return iter(self.order)

    def __getitem__(self, idx):
        return self.order[idx]

    def __len__(self):
        return len(self.order)

## b. Implemente estructuras de colas FIFO, LIFO y PRIORITY con los siguientes métodos:

- EMPTY: retorna TRUE solamente si la cola se encuentra vacía.
- TOP: retorna el primer elemento de la lista.
- POP: retorna el primer elemento de la lista Y lo remueve de la lista.
- ADD: Inserta 1 elemento en la lista y retorna la lista con el elemento en su posición correspondiente.

In [8]:
class FIFO:
    class _QueueNode:
        def __init__(self, value):
            self.value = value   # TreeNode
            self.next = None

    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def EMPTY(self):
        return self.size == 0

    def TOP(self):
        if self.EMPTY():
            return None
        return self.head.value

    def POP(self):
        if self.EMPTY():
            return None
        front = self.head.value
        self.head = self.head.next
        if self.head is None:
            self.tail = None
        self.size -= 1
        return front

    def ADD(self, element):
        # element should be a TreeNode
        new_node = self._QueueNode(element)
        if self.EMPTY():
            self.head = self.tail = new_node
        else:
            self.tail.next = new_node
            self.tail = new_node
        self.size += 1
        return self


class LIFO:
    class _QueueNode:
        def __init__(self, value):
            self.value = value
            self.next = None

    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def EMPTY(self):
        return self.size == 0

    def TOP(self):
        if self.EMPTY():
            return None
        return self.tail.value

    def POP(self):
        if self.EMPTY():
            return None

        value = self.tail.value

        # Single element case
        if self.head == self.tail:
            self.head = None
            self.tail = None
            self.size -= 1
            return value

        # Find node before tail
        current = self.head
        while current.next != self.tail:
            current = current.next

        current.next = None
        self.tail = current
        self.size -= 1
        return value

    def ADD(self, element):
        new_node = self._QueueNode(element)
        if self.EMPTY():
            self.head = self.tail = new_node
        else:
            self.tail.next = new_node
            self.tail = new_node
        self.size += 1
        return self


class Priority:
    class _QueueNode:
        def __init__(self, value):
            self.value = value   # TreeNode
            self.next = None

    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def EMPTY(self):
        return self.size == 0

    def TOP(self):
        if self.EMPTY():
            return None
        return self.head.value

    def POP(self):
        if self.EMPTY():
            return None
        front = self.head.value
        self.head = self.head.next
        if self.head is None:
            self.tail = None
        self.size -= 1
        return front

    def ADD(self, element):
        new_node = self._QueueNode(element)

        if self.EMPTY():
            self.head = self.tail = new_node
        elif element.heuristic < self.head.value.heuristic:
            new_node.next = self.head
            self.head = new_node
        else:
            current = self.head
            while current.next and current.next.value.heuristic <= element.heuristic:
                current = current.next
            new_node.next = current.next
            current.next = new_node
            if new_node.next is None:
                self.tail = new_node

        self.size += 1
        return self


## c. Implemente lo 5 algoritmos, tomando como inputs las funciones de costo.


In [10]:
Heuristica = openpyxl.load_workbook("files/Heuristica.xlsx")
Heuristica = Heuristica.active
# remove the first row
Heuristica.delete_rows(1)

Costo_Funciones = openpyxl.load_workbook("files/Costo_Function.xlsx")
Costo_Funciones = Costo_Funciones.active
Costo_Funciones.delete_rows(1)

# for row in range(0, Heuristica.max_row):
#     for col in Heuristica.iter_cols(1, Heuristica.max_column):
#         print(col[row].value)

# for row in range(0, Costo_Funciones.max_row):
#     for col in Costo_Funciones.iter_cols(1, Costo_Funciones.max_column):
#         print(col[row].value)


### Tree

In [11]:
#  Make tree with excel files readed


tree = Tree()

# Crear nodos desde Heuristica
for row in range(0, Heuristica.max_row):
    node_name = Heuristica.cell(row=row + 1, column=1).value
    heuristic_value = Heuristica.cell(row=row + 1, column=2).value
    tree.add_node(node_name, heuristic_value)

# Crear aristas desde Costo_Funciones
for row in range(0, Costo_Funciones.max_row):
    parent_name = Costo_Funciones.cell(row=row + 1, column=1).value
    child_name = Costo_Funciones.cell(row=row + 1, column=2).value
    cost_value = Costo_Funciones.cell(row=row + 1, column=3).value
    tree.add_edge(parent_name, child_name, cost_value)

tree.set_root()


print("Tree structure:")
for node in tree:
    print(f"Node: {node.name}, Heuristic: {node.heuristic}, parent: {node.parent.name if node.parent else None}")
    for child, cost in zip(node.children, node.cost):
        print(f"  Child: {child.name}, Cost: {cost}")


Tree structure:
Node: Warm-up activities, Heuristic: 5, parent: None
  Child: Skipping Rope, Cost: 10
  Child: Exercise bike, Cost: 10
  Child: Tread Mill, Cost: 10
  Child: Step Mill, Cost: 10
Node: Skipping Rope, Heuristic: 16, parent: Warm-up activities
  Child: Dumbbell, Cost: 15
  Child: Barbell, Cost: 15
Node: Exercise bike, Heuristic: 10, parent: Warm-up activities
  Child: Cable-Crossover, Cost: 25
Node: Tread Mill, Heuristic: 12, parent: Warm-up activities
  Child: Pulling Bars, Cost: 20
  Child: Incline Bench, Cost: 20
Node: Step Mill, Heuristic: 14, parent: Warm-up activities
  Child: Incline Bench, Cost: 16
Node: Dumbbell, Heuristic: 9, parent: Skipping Rope
  Child: Leg Press Machine, Cost: 12
Node: Barbell, Heuristic: 10, parent: Skipping Rope
  Child: Leg Press Machine, Cost: 10
Node: Cable-Crossover, Heuristic: 8, parent: Exercise bike
  Child: Climbing Rope, Cost: 10
Node: Pulling Bars, Heuristic: 10, parent: Tread Mill
  Child: Climbing Rope, Cost: 6
Node: Incline Ben

### 1. Breadth-first search

In [12]:
def bfs(nodes, initial_node, goal_node):
    if not nodes:
        return []

    q = FIFO()
    visited = set()
    parent = {}

    visited.add(initial_node.name)
    parent[initial_node.name] = None
    q.ADD(initial_node)

    while not q.EMPTY():
        curr = q.POP()

        if curr.name == goal_node.name:
            path = []
            node_name = curr.name
            while node_name is not None:
                path.append(node_name)
                node_name = parent[node_name]
            path.reverse()
            return path

        for child in curr.children:
            if child.name not in visited:
                visited.add(child.name)
                parent[child.name] = curr.name
                q.ADD(child)

    return []

bfs_result = bfs(tree, tree[0], tree[-1])
print("BFS Result:", bfs_result)



BFS Result: ['Warm-up activities', 'Skipping Rope', 'Dumbbell', 'Leg Press Machine', 'Stretching']


### 3. Uniform-cost search

In [13]:
def UCS(nodes, initial_node, goal_node):
    if not nodes or initial_node is None or goal_node is None:
        return []

    frontier = [(0, initial_node.name, initial_node)]
    parent = {initial_node.name: None}
    cost_so_far = {initial_node.name: 0}
    visited = set()

    while frontier:
        frontier.sort(key=lambda x: (x[0], x[1]))
        current_cost, _, curr = frontier.pop(0)

        # Skip outdated entries
        if current_cost > cost_so_far.get(curr.name, float('inf')):
            continue

        if curr.name == goal_node.name:
            path = []
            node_name = curr.name
            while node_name is not None:
                path.append(node_name)
                node_name = parent[node_name]
            path.reverse()
            return path

        if curr.name in visited:
            continue
        visited.add(curr.name)

        for child, edge_cost in zip(curr.children, curr.cost):
            new_cost = current_cost + edge_cost
            if new_cost < cost_so_far.get(child.name, float('inf')):
                cost_so_far[child.name] = new_cost
                parent[child.name] = curr.name
                frontier.append((new_cost, child.name, child))

    return []

ucs_result = UCS(tree, tree[0], tree[-1])
print("UCS Result:", ucs_result)

UCS Result: ['Warm-up activities', 'Skipping Rope', 'Barbell', 'Leg Press Machine', 'Stretching']


### 4. Greedy Best-first search

In [16]:
def gbfs(nodes, initial_node, goal_node):
    if not nodes:
        return []

    q = Priority()
    visited = set()
    path = []

    q.ADD(initial_node)

    while not q.EMPTY():
        curr = q.POP()
        path.append(curr.name)

        if curr.name == goal_node.name:
            return path

        visited.add(curr.name)

        for child in curr.children:
            if child.name not in visited:
                q.ADD(child)

    return []

gbfs_result = gbfs(tree, tree[0], tree[-1])
print("GBFS Result:", gbfs_result)

GBFS Result: ['Warm-up activities', 'Exercise bike', 'Cable-Crossover', 'Climbing Rope', 'Stretching']


### 5. A-Star

In [18]:
def astar(nodes, initial_node, goal_node):
    if not nodes:
        return []

    q = Priority()
    visited = set()
    parent = {}
    g_cost = {}

    g_cost[initial_node.name] = 0
    initial_node.heuristic = initial_node.heuristic  # f = g + h (g=0)

    q.ADD(initial_node)
    parent[initial_node.name] = None

    while not q.EMPTY():
        curr = q.POP()

        if curr.name == goal_node.name:
            path = []
            node = curr.name
            while node:
                path.append(node)
                node = parent[node]
            return path[::-1]

        visited.add(curr.name)

        for child in curr.children:
            tentative_g = g_cost[curr.name] + 1

            if child.name not in g_cost or tentative_g < g_cost[child.name]:
                g_cost[child.name] = tentative_g
                f = tentative_g + child.heuristic

                child.heuristic = f  # priority = f(n)

                parent[child.name] = curr.name
                q.ADD(child)

    return []

gbfs_result = astar(tree, tree[0], tree[-1])
print("GBFS Result:", gbfs_result)

GBFS Result: ['Warm-up activities', 'Exercise bike', 'Cable-Crossover', 'Climbing Rope', 'Stretching']
